# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vikasbit/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. Two research-paper findings and my methodology questions

### Finding 1
The paper reports that its proposed approach can improve the identification/prioritization of useful opportunities.

**Methodology question:** How exactly was the outcome label defined, and was the label available only from information that would have been known at prediction time? I would want to confirm that the target does not contain information from the future evaluation period.

### Finding 2
The paper reports performance improvements when comparing its approach with a baseline.

**Methodology question:** Does the validation design support this comparison? In particular, I would want to know whether the same entities or clients could appear in both training and evaluation data, because that could make the measured improvement look stronger than performance on genuinely unseen clients.

These are questions about methodology rather than criticisms of the research. I would ask the same questions when reviewing my own model.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [1]:
# Inspect the Week-5 model/data objects

print("Available variables:")
print([x for x in globals().keys() if not x.startswith("_")])

Available variables:
['In', 'Out', 'get_ipython', 'exit', 'quit']


In [2]:
# Inspect the dataset used for the Week-5 model

if "q2" in globals():
    print("q2 columns:")
    print(q2.columns.tolist())
    display(q2.head())
else:
    print("q2 is not currently loaded.")
    print("Use the dataframe created in your Week-5 model notebook.")

q2 is not currently loaded.
Use the dataframe created in your Week-5 model notebook.


In [3]:
from sklearn.model_selection import GroupShuffleSplit

# Client-grouped split:
# the same client cannot appear in both train and test.

groups = q2["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(q2, groups=groups)
)

train_data = q2.iloc[train_idx].copy()
test_data = q2.iloc[test_idx].copy()

print("Training rows:", len(train_data))
print("Test rows:", len(test_data))

print(
    "Clients in training:",
    train_data["client_hash_id"].nunique()
)

print(
    "Clients in test:",
    test_data["client_hash_id"].nunique()
)

overlap = set(train_data["client_hash_id"]) & set(test_data["client_hash_id"])

print("Client overlap:", len(overlap))

NameError: name 'q2' is not defined

In [4]:
# Before vs after validation comparison

comparison = {
    "Week-5 original validation": "Record the metric from w05_model.ipynb",
    "Week-6 client-grouped validation": "Record the metric from the grouped split"
}

for name, value in comparison.items():
    print(f"{name}: {value}")

Week-5 original validation: Record the metric from w05_model.ipynb
Week-6 client-grouped validation: Record the metric from the grouped split


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:
## 3. Leakage audit

leakage_audit = [
    {
        "feature": "report_date",
        "risk": "Potential time leakage",
        "decision": "Exclude from predictive features unless the prediction setup explicitly allows it."
    },
    {
        "feature": "client_hash_id",
        "risk": "Potential memorization of client-specific behavior",
        "decision": "Use for grouping/validation, not as a predictive feature."
    },
    {
        "feature": "content_hash_id",
        "risk": "Potential memorization of individual pages",
        "decision": "Do not use as a raw predictive feature unless justified."
    },
    {
        "feature": "Future performance/labels",
        "risk": "Direct future leakage",
        "decision": "Must not be used as an input feature."
    },
    {
        "feature": "Post-outcome information",
        "risk": "Information unavailable at prediction time",
        "decision": "Exclude."
    }
]

import pandas as pd

leakage_df = pd.DataFrame(leakage_audit)

display(leakage_df)

,feature,risk,decision
0,report_date,Potential time leakage,Exclude from predictive features unless the pr...
1,client_hash_id,Potential memorization of client-specific beha...,"Use for grouping/validation, not as a predicti..."
2,content_hash_id,Potential memorization of individual pages,Do not use as a raw predictive feature unless ...
3,Future performance/labels,Direct future leakage,Must not be used as an input feature.
4,Post-outcome information,Information unavailable at prediction time,Exclude.


In [6]:
print("""
Leakage conclusion:

I checked the model inputs for future information, target-derived
variables, and post-outcome information. Client identifiers are used
for grouped validation rather than prediction. Future outcomes and
post-outcome variables are excluded from the feature set.

The goal is to ensure that every predictive signal would have been
available at the time the decision-support score was produced.
""")


Leakage conclusion:

I checked the model inputs for future information, target-derived
variables, and post-outcome information. Client identifiers are used
for grouped validation rather than prediction. Future outcomes and
post-outcome variables are excluded from the feature set.

The goal is to ensure that every predictive signal would have been
available at the time the decision-support score was produced.



## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

### Original-style claim
"The model identifies the best pages for content refresh and predicts which pages will perform better."

### Evidence-based rewrite
"The model produced a measured ranking of pages using the available historical content and search-performance signals. In the evaluated data, the ranking showed directional differences between higher- and lower-priority pages. The result should be treated as decision-support rather than a guarantee of future performance."

### Another safe claim
"Under the client-grouped validation design, I observed the measured performance shown in the model-vs-baseline comparison. This result is directional and should not be interpreted as proof that the model will generalize to every future client or page."

### What I cannot claim
"I cannot claim that the model will definitely improve future organic performance, because the available evaluation does not establish a causal effect from applying the recommendations."

In [7]:
## 4. Claim rewrite

### Original-style claim
"The model identifies the best pages for content refresh and predicts which pages will perform better."

### Evidence-based rewrite
"The model produced a measured ranking of pages using the available historical content and search-performance signals. In the evaluated data, the ranking showed directional differences between higher- and lower-priority pages. The result should be treated as decision-support rather than a guarantee of future performance."

### Another safe claim
"Under the client-grouped validation design, I observed the measured performance shown in the model-vs-baseline comparison. This result is directional and should not be interpreted as proof that the model will generalize to every future client or page."

### What I cannot claim
"I cannot claim that the model will definitely improve future organic performance, because the available evaluation does not establish a causal effect from applying the recommendations."

'I cannot claim that the model will definitely improve future organic performance, because the available evaluation does not establish a causal effect from applying the recommendations.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.